In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!cp -r /content/drive/MyDrive/split_dataset /content/

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

cuda
Tesla T4


In [ ]:
train_dir = '/content/split_dataset/train'
val_dir = '/content/split_dataset/val'

print(os.path.exists(train_dir), train_dir)
print(os.path.exists(val_dir), val_dir)

True /content/split_dataset/train
True /content/split_dataset/val


In [ ]:
img_size = 224
batch_size = 32

train_tfms = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
])

val_tfms = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
])

In [ ]:
train_ds = datasets.ImageFolder(train_dir, transform=train_tfms)
val_ds = datasets.ImageFolder(val_dir, transform=val_tfms)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)

class_names = train_ds.classes
print(class_names)
print("Train samples:", len(train_ds))
print("Val samples:", len(val_ds))

['destroyed', 'major-damage', 'minor-damage', 'no-damage']
Train samples: 6270
Val samples: 1570


In [ ]:
class HybridCNNResNet(nn.Module):
    def __init__(self, num_classes=4):
        super(HybridCNNResNet, self).__init__()

        self.cnn_branch = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.AdaptiveAvgPool2d((1, 1))
        )

        resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        self.resnet_branch = nn.Sequential(*list(resnet.children())[:-1])

        self.classifier = nn.Sequential(
            nn.Linear(128 + 2048, 512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        cnn_feat = self.cnn_branch(x)
        cnn_feat = torch.flatten(cnn_feat, 1)

        resnet_feat = self.resnet_branch(x)
        resnet_feat = torch.flatten(resnet_feat, 1)

        fused = torch.cat((cnn_feat, resnet_feat), dim=1)
        out = self.classifier(fused)
        return out

In [ ]:
model = HybridCNNResNet(num_classes=4).to(device)
print(model)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 136MB/s]


HybridCNNResNet(
  (cnn_branch): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (9): AdaptiveAvgPool2d(output_size=(1, 1))
  )
  (resnet_branch): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): Bottleneck(
        (conv1): 

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss = 0.0
    all_preds, all_labels = [], []

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average='macro')

    return epoch_loss, epoch_acc, epoch_f1


def validate_one_epoch(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average='macro')

    return epoch_loss, epoch_acc, epoch_f1, all_labels, all_preds

In [ ]:
num_epochs = 5
best_f1 = 0.0
save_path = '/content/best_hybrid_cnn_resnet_imbalanced.pth'

for epoch in range(num_epochs):
    train_loss, train_acc, train_f1 = train_one_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc, val_f1, y_true, y_pred = validate_one_epoch(model, val_loader, criterion)

    scheduler.step(val_f1)

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), save_path)

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Train F1: {train_f1:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f} | Val   F1: {val_f1:.4f}")
    print("-" * 60)

Epoch 1/5
Train Loss: 0.7413 | Train Acc: 0.7225 | Train F1: 0.4434
Val   Loss: 0.5671 | Val   Acc: 0.7924 | Val   F1: 0.6380
------------------------------------------------------------
Epoch 2/5
Train Loss: 0.5528 | Train Acc: 0.8019 | Train F1: 0.6672
Val   Loss: 0.4617 | Val   Acc: 0.8217 | Val   F1: 0.7075
------------------------------------------------------------
Epoch 3/5
Train Loss: 0.4676 | Train Acc: 0.8271 | Train F1: 0.7186
Val   Loss: 0.4452 | Val   Acc: 0.8255 | Val   F1: 0.7296
------------------------------------------------------------
Epoch 4/5
Train Loss: 0.4048 | Train Acc: 0.8470 | Train F1: 0.7514
Val   Loss: 0.4029 | Val   Acc: 0.8484 | Val   F1: 0.7485
------------------------------------------------------------
Epoch 5/5
Train Loss: 0.3557 | Train Acc: 0.8683 | Train F1: 0.7835
Val   Loss: 0.4336 | Val   Acc: 0.8478 | Val   F1: 0.7661
------------------------------------------------------------


In [ ]:
model.load_state_dict(torch.load(save_path))

_, val_acc, val_f1, y_true, y_pred = validate_one_epoch(model, val_loader, criterion)

precision = precision_score(y_true, y_pred, average='macro')
recall = recall_score(y_true, y_pred, average='macro')
f1 = f1_score(y_true, y_pred, average='macro')

print("Best Validation Accuracy:", val_acc)
print("Best Validation Precision:", precision)
print("Best Validation Recall:", recall)
print("Best Validation Macro F1:", f1)

print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, target_names=class_names))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_true, y_pred))

Best Validation Accuracy: 0.8477707006369427
Best Validation Precision: 0.7579137150977293
Best Validation Recall: 0.7792487856605779
Best Validation Macro F1: 0.7660528960223638

Classification Report:

              precision    recall  f1-score   support

   destroyed       0.87      0.90      0.88       130
major-damage       0.62      0.77      0.69       219
minor-damage       0.60      0.54      0.57       169
   no-damage       0.94      0.91      0.92      1052

    accuracy                           0.85      1570
   macro avg       0.76      0.78      0.77      1570
weighted avg       0.85      0.85      0.85      1570


Confusion Matrix:

[[117   5   5   3]
 [  5 169  21  24]
 [  4  43  91  31]
 [  9  54  35 954]]


In [ ]:
print("\nFINAL METRICS")
print(f"Accuracy  : {val_acc:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")


FINAL METRICS
Accuracy  : 0.8478
Precision : 0.7579
Recall    : 0.7792
F1 Score  : 0.7661
